# 01 · EDA and RCT analysis, the Hillstrom e-mail experiment

**Business question.** A retailer can e-mail any of its customers, but an
e-mail only *pays off* when it changes behaviour. Classic response models
target customers who are likely to buy, including the "sure things" who
would have bought anyway, and may even contact "do-not-disturbs" who react
negatively. Uplift modelling instead targets **persuadables**: customers
whose behaviour the e-mail actually changes.

This notebook covers the foundation: exploring the data and analysing the
randomised experiment itself. Everything here re-uses the functions in
`src/`, so the pipeline scripts and the notebook can never drift apart.

**The dataset** (Hillstrom, 2008, the MineThatData e-mail analytics
challenge): 64,000 customers who purchased in the previous twelve months,
randomised 1:1:1 to a Men's-merchandise e-mail, a Women's-merchandise
e-mail, or no e-mail, then tracked for two weeks:

| column | meaning |
|---|---|
| `recency` | months since last purchase (1-12) |
| `history_segment`, `history` | past-year spend band / dollar value |
| `mens`, `womens` | bought men's / women's merchandise in the past year |
| `zip_code` | Suburban / Rural / Urban |
| `newbie` | new customer in the past twelve months |
| `channel` | Phone / Web / Multichannel |
| `segment` | treatment arm |
| `visit`, `conversion`, `spend` | outcomes over the following two weeks |

In [ ]:
import sys

sys.path.append("..")

import pandas as pd

from src import config
from src.data_ingestion import load_clean
from src.rct_analysis import (
    ate_table,
    balance_table,
    plot_segments,
    segment_ates,
    srm_check,
)

pd.set_option("display.width", 140)
df = load_clean()
df.head()

## The experiment at a glance

Three arms of ~21,300 customers each. `visit` is the highest-signal outcome
(~10.6% base rate); `conversion` is rare (~0.6%) and `spend` is zero-inflated
and heavy-tailed, this ordering will matter for every standard error in the
project.

In [ ]:
print(df["treatment"].value_counts().to_string(), "\n")
df.groupby("treatment")[config.OUTCOMES].mean().round(4)

## Covariates

`history` is heavily right-skewed (hence the engineered `log_history`), and
roughly half the customers are `newbie`s. All of these variables are
*pre-treatment*, which is what makes them legitimate adjustment variables in
Part 2 and legitimate effect-modifiers for the uplift models in Part 3.

In [ ]:
display(df[config.FEATURES].describe().T.round(3))
df["history_segment"].value_counts().sort_index()

## Is the randomisation sound?

Two checks before trusting difference-in-means:

1. **Sample-ratio mismatch (SRM)**: chi-square test of the realised arm
   sizes against the intended 1:1:1 split. SRM is one of the most common
   silent failures in real experimentation platforms.
2. **Covariate balance**: standardised mean differences
   $\mathrm{SMD} = (\bar{x}_t - \bar{x}_c) / s_{pooled}$ for every
   covariate, each arm vs control. |SMD| < 0.1 is the usual rule of thumb.

In [ ]:
print(srm_check(df))
balance_table(df).round(4)

On the real data the SRM p-value is ≈ 0.90 and the largest |SMD| is ≈ 0.014,
the randomisation is clean, so a difference in means identifies the ATE:

$$\widehat{\mathrm{ATE}} = \bar{Y}_{treat} - \bar{Y}_{control},$$

tested with a Welch t-test (unequal variances) and, in parallel, a
percentile bootstrap CI that makes no variance-formula assumptions, a
useful cross-check for the skewed `spend` outcome.

Because there are **2 treatments × 3 outcomes = 6 hypotheses**, raw p-values
are Holm-adjusted (step-down; controls FWER without independence
assumptions).

In [ ]:
ate = ate_table(df)
ate[["treatment", "outcome", "mean_treat", "mean_ctrl", "ate",
     "rel_lift_pct", "p_raw", "p_holm", "ci_boot_lo", "ci_boot_hi",
     "significant_5pct"]]

All six effects survive Holm correction. Headline numbers on the real data:
the Men's e-mail lifts visits by **+7.7pp** and the Women's e-mail by
**+4.5pp** (vs a 10.6% control base rate); conversion and spend lifts are
positive but estimated far less precisely, exactly why Part 3 models
`visit` and Part 4 prices conversions separately.

## Heterogeneity, the reason this project exists

If every customer reacted identically, the analysis would end here: e-mail
everyone whenever the margin covers the cost. Segment-level ATEs say
otherwise.

In [ ]:
seg = segment_ates(df)
plot_segments(seg)  # also saved to reports/figures/
seg[seg["treatment"] == "womens"]

The Women's-e-mail effect on visits ranges from ~3.6pp to ~6.6pp across
history segments and channels, nearly a 2× spread from one-dimensional
cuts alone, and multivariate models will widen it (Part 3's top decile
reaches +8.5pp). Averages hide who is actually persuadable. That motivates
the rest of the project:

* **02 · Part 2**: if we *couldn't* randomise, could we still recover these
  effects? (Yes, and we can prove it, because here the truth is known.)
* **02 · Part 3**: model the effect *per customer* with uplift
  meta-learners.
* **02 · Part 4**: turn those scores into a profit-optimal targeting rule.